# GNN-Based Dynamic Pricing Model

This notebook implements a Graph Neural Network for dynamic pricing optimization using Snowflake's ML Container Runtime with GPU support.

**Architecture:**
- **Snowflake Feature Store** for centralized feature management (product, store, sales aggregates)
- **Heterogeneous GNN** using pure PyTorch with 4 edge types (product-product, store-store, product-store, store-product)
- **Distributed training** via Snowflake's `PyTorchDistributor` with `ShardedDataConnector`

**Data Model:** `DYNAMIC_PRICING.PRICING_MODEL` with 200 products, 50 stores, 365 days of sales data

### Steps
1. Setup & Imports
2. Feature Store Creation (entities, feature views)
3. Spine DataFrame & Training Data
4. Graph Construction (heterogeneous graph)
5. GNN Model Definition
6. Distributed Training with PyTorchDistributor

## 1. Setup

### GPU Device Info
Verify GPU availability in the Container Runtime environment.

In [ ]:
import torch

if torch.cuda.is_available():
    num_gpus = torch.cuda.device_count()
    print(f"GPU devices available: {num_gpus}")
    for i in range(num_gpus):
        print(f"  Device {i}: {torch.cuda.get_device_name(i)}")
    torch.cuda.set_device(0)
else:
    print("CUDA is not available. Check your GPU setup.")

In [ ]:
import os
import time
import json
import pickle

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader

from snowflake.ml.modeling.preprocessing import LabelEncoder, MinMaxScaler
from snowflake.ml.modeling.pipeline import Pipeline
from snowflake.ml.modeling.distributors.pytorch import (
    PyTorchDistributor,
    PyTorchScalingConfig,
    WorkerResourceConfig,
)
from snowflake.ml.modeling.distributors.pytorch import get_context
from snowflake.ml.data.sharded_data_connector import ShardedDataConnector
from snowflake.ml.feature_store import (
    FeatureStore,
    FeatureView,
    Entity,
    CreationMode,
)
from snowflake.snowpark import functions as F
from snowflake.snowpark import types as T
from snowflake.snowpark.context import get_active_session

import warnings
warnings.filterwarnings('ignore')

session = get_active_session()
session.query_tag = {
    "origin": "sf_sit",
    "name": "dynamic_pricing_gnn",
    "version": {"major": 1, "minor": 0},
}

DB = "DYNAMIC_PRICING"
DATA_SCHEMA = "PRICING_MODEL"
FS_SCHEMA = "FS_SCHEMA"
WH = str(session.get_current_warehouse()).strip('"')

print(f"Database: {DB}")
print(f"Data Schema: {DATA_SCHEMA}")
print(f"Feature Store Schema: {FS_SCHEMA}")
print(f"Warehouse: {WH}")

## 2. Feature Store Creation

Create a Feature Store in a dedicated schema within the `DYNAMIC_PRICING` database. We register entities and feature views for products, stores, and aggregated sales metrics.

In [ ]:
# Create the Feature Store schema
session.sql(f"CREATE SCHEMA IF NOT EXISTS {DB}.{FS_SCHEMA}").collect()
session.sql(f"USE SCHEMA {DB}.{FS_SCHEMA}").collect()
print(f"Feature Store schema ready: {DB}.{FS_SCHEMA}")

In [ ]:
# Initialize the Feature Store
fs = FeatureStore(
    session=session,
    database=DB,
    name=FS_SCHEMA,
    default_warehouse=WH,
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST,
)
print("Feature Store initialized.")

### Feature DataFrames
Build Snowpark DataFrames that define the features for each entity.

In [ ]:
# Product features from DIM_PRODUCT
product_features_df = session.sql(f"""
    SELECT
        PRODUCT_ID,
        CATEGORY,
        SUBCATEGORY,
        BRAND,
        COST,
        BASE_PRICE,
        WEIGHT
    FROM {DB}.{DATA_SCHEMA}.DIM_PRODUCT
""")
product_features_df.show(5)

In [ ]:
# Store features from DIM_STORE
store_features_df = session.sql(f"""
    SELECT
        STORE_ID,
        REGION,
        CITY,
        STORE_TYPE,
        LAT,
        LON
    FROM {DB}.{DATA_SCHEMA}.DIM_STORE
""")
store_features_df.show(5)

In [ ]:
# Aggregated sales features per product-store pair
sales_agg_features_df = session.sql(f"""
    SELECT
        PRODUCT_ID,
        STORE_ID,
        ROUND(AVG(SELLING_PRICE), 2)                             AS AVG_SELLING_PRICE,
        ROUND(AVG(UNITS_SOLD), 2)                                AS AVG_UNITS_SOLD,
        ROUND(AVG(REVENUE), 2)                                   AS AVG_REVENUE,
        ROUND(AVG(SELLING_PRICE - COST_AT_SALE), 2)              AS AVG_MARGIN,
        ROUND(STDDEV(SELLING_PRICE), 2)                          AS PRICE_VOLATILITY,
        ROUND(SUM(CASE WHEN INVENTORY_START_OF_DAY = 0 THEN 1 ELSE 0 END)
              / COUNT(*), 4)                                     AS STOCKOUT_RATE
    FROM {DB}.{DATA_SCHEMA}.FACT_SALES_DAILY
    GROUP BY PRODUCT_ID, STORE_ID
""")
print(f"Sales aggregate features: {sales_agg_features_df.count()} product-store pairs")
sales_agg_features_df.show(5)

### Register Entities

In [ ]:
# Product entity
product_entity = Entity(
    name="PRODUCT",
    join_keys=["PRODUCT_ID"],
    desc="Product dimension entity",
)
fs.register_entity(product_entity)

# Store entity
store_entity = Entity(
    name="STORE",
    join_keys=["STORE_ID"],
    desc="Store dimension entity",
)
fs.register_entity(store_entity)

# Product-Store entity (composite key for sales aggregates)
product_store_entity = Entity(
    name="PRODUCT_STORE",
    join_keys=["PRODUCT_ID", "STORE_ID"],
    desc="Product-Store composite entity for sales features",
)
fs.register_entity(product_store_entity)

fs.list_entities().show()

### Register Feature Views

In [ ]:
# Product Features View
product_fv = FeatureView(
    name="PRODUCT_FEATURES",
    entities=[product_entity],
    feature_df=product_features_df,
    refresh_freq="1 day",
    desc="Product attributes from DIM_PRODUCT",
)
product_fv = fs.register_feature_view(
    feature_view=product_fv,
    version="V1",
    block=True,
    overwrite=True,
)
print("Registered PRODUCT_FEATURES V1")

# Store Features View
store_fv = FeatureView(
    name="STORE_FEATURES",
    entities=[store_entity],
    feature_df=store_features_df,
    refresh_freq="1 day",
    desc="Store attributes from DIM_STORE",
)
store_fv = fs.register_feature_view(
    feature_view=store_fv,
    version="V1",
    block=True,
    overwrite=True,
)
print("Registered STORE_FEATURES V1")

# Sales Aggregate Features View
sales_agg_fv = FeatureView(
    name="SALES_AGG_FEATURES",
    entities=[product_store_entity],
    feature_df=sales_agg_features_df,
    refresh_freq="1 day",
    desc="Aggregated sales metrics per product-store pair",
)
sales_agg_fv = fs.register_feature_view(
    feature_view=sales_agg_fv,
    version="V1",
    block=True,
    overwrite=True,
)
print("Registered SALES_AGG_FEATURES V1")

fs.list_feature_views().select("NAME", "VERSION", "DESC").show()

## 3. Spine DataFrame & Training Data

The spine defines the set of (product_id, store_id) training examples. We use the Feature Store to join features onto the spine and build our training dataset.

**Label:** `SELLING_PRICE` (average daily selling price per product-store pair) -- the GNN learns to predict optimal pricing.

In [ ]:
# Build spine: unique product-store pairs with their average selling price as the label
spine_df = session.sql(f"""
    SELECT
        PRODUCT_ID,
        STORE_ID,
        ROUND(AVG(SELLING_PRICE), 2) AS TARGET_PRICE
    FROM {DB}.{DATA_SCHEMA}.FACT_SALES_DAILY
    GROUP BY PRODUCT_ID, STORE_ID
""")
print(f"Spine size: {spine_df.count()} product-store pairs")
spine_df.show(5)

In [ ]:
# Generate dataset from Feature Store by joining features onto spine
training_dataset = fs.generate_dataset(
    name="PRICING_TRAINING_DATA",
    spine_df=spine_df,
    features=[product_fv, store_fv, sales_agg_fv],
)

full_df = training_dataset.read.to_snowpark_dataframe()
print(f"Full dataset: {full_df.count()} rows, {len(full_df.columns)} columns")
print(f"Columns: {full_df.columns}")
full_df.show(5)

In [ ]:
# Define feature groups for preprocessing
sparse_features = [
    "CATEGORY",
    "SUBCATEGORY",
    "BRAND",
    "REGION",
    "CITY",
    "STORE_TYPE",
]

dense_features = [
    "COST",
    "BASE_PRICE",
    "WEIGHT",
    "LAT",
    "LON",
    "AVG_SELLING_PRICE",
    "AVG_UNITS_SOLD",
    "AVG_REVENUE",
    "AVG_MARGIN",
    "PRICE_VOLATILITY",
    "STOCKOUT_RATE",
]

label_col = "TARGET_PRICE"
id_cols = ["PRODUCT_ID", "STORE_ID"]

print(f"Sparse features ({len(sparse_features)}): {sparse_features}")
print(f"Dense features ({len(dense_features)}): {dense_features}")
print(f"Label: {label_col}")

In [ ]:
# Preprocessing pipeline: LabelEncode categoricals, MinMaxScale numericals
pipeline_steps = []

for i, feat in enumerate(sparse_features):
    pipeline_steps.append((
        f"LE_{feat}",
        LabelEncoder(input_cols=[feat], output_cols=[feat]),
    ))

pipeline_steps.append((
    "MMS",
    MinMaxScaler(
        feature_range=(0, 1),
        input_cols=dense_features,
        output_cols=dense_features,
    ),
))

preprocessing_pipeline = Pipeline(steps=pipeline_steps)
preprocessed_df = preprocessing_pipeline.fit(full_df).transform(full_df)

print(f"Preprocessed dataset: {preprocessed_df.count()} rows")
preprocessed_df.show(5)

In [ ]:
# Train/validation split (80/20) and save as temp tables
train_df, val_df = preprocessed_df.random_split([0.8, 0.2], seed=42)

train_df.write.mode("overwrite").save_as_table(
    f"{DB}.{DATA_SCHEMA}.TRAIN_DATA", table_type="temporary"
)
val_df.write.mode("overwrite").save_as_table(
    f"{DB}.{DATA_SCHEMA}.VAL_DATA", table_type="temporary"
)

train_count = session.table(f"{DB}.{DATA_SCHEMA}.TRAIN_DATA").count()
val_count = session.table(f"{DB}.{DATA_SCHEMA}.VAL_DATA").count()
print(f"Train: {train_count} rows | Val: {val_count} rows")

## 4. Graph Construction

Build a heterogeneous graph from the edge tables:
- **Product-Product edges** (substitute, complement, cannibalization, same_category)
- **Store-Store edges** (geographic proximity)
- **Product-Store edges** (which products are sold in which stores)

Node features come from the Feature Store (product and store attributes). The graph structure is serialized so it can be reconstructed inside the distributed training function.

In [ ]:
# Load edge tables
pp_edges_pdf = session.sql(f"""
    SELECT SRC_PRODUCT_ID, DST_PRODUCT_ID, WEIGHT
    FROM {DB}.{DATA_SCHEMA}.GRAPH_PRODUCT_PRODUCT_EDGE
    WHERE SNAPSHOT_DATE = '2025-01-01'
""").to_pandas()

ss_edges_pdf = session.sql(f"""
    SELECT SRC_STORE_ID, DST_STORE_ID, WEIGHT
    FROM {DB}.{DATA_SCHEMA}.GRAPH_STORE_STORE_EDGE
    WHERE SNAPSHOT_DATE = '2025-01-01'
""").to_pandas()

ps_edges_pdf = session.sql(f"""
    SELECT PRODUCT_ID, STORE_ID, WEIGHT
    FROM {DB}.{DATA_SCHEMA}.GRAPH_PRODUCT_STORE_EDGE
    WHERE SNAPSHOT_DATE = '2025-01-01'
""").to_pandas()

print(f"Product-Product edges: {len(pp_edges_pdf)}")
print(f"Store-Store edges:     {len(ss_edges_pdf)}")
print(f"Product-Store edges:   {len(ps_edges_pdf)}")

In [ ]:
# Build node feature matrices from the Feature Store
# Product node features: encode categoricals, scale numericals
product_pdf = session.sql(f"""
    SELECT PRODUCT_ID, CATEGORY, SUBCATEGORY, BRAND, COST, BASE_PRICE, WEIGHT
    FROM {DB}.{DATA_SCHEMA}.DIM_PRODUCT
    ORDER BY PRODUCT_ID
""").to_pandas()

store_pdf = session.sql(f"""
    SELECT STORE_ID, REGION, CITY, STORE_TYPE, LAT, LON
    FROM {DB}.{DATA_SCHEMA}.DIM_STORE
    ORDER BY STORE_ID
""").to_pandas()

# Encode product categorical features as integers
for col in ["CATEGORY", "SUBCATEGORY", "BRAND"]:
    product_pdf[col] = product_pdf[col].astype("category").cat.codes.astype(float)

# Encode store categorical features as integers
for col in ["REGION", "CITY", "STORE_TYPE"]:
    store_pdf[col] = store_pdf[col].astype("category").cat.codes.astype(float)

# Normalize numeric columns to [0, 1]
for col in ["COST", "BASE_PRICE", "WEIGHT"]:
    cmin, cmax = product_pdf[col].min(), product_pdf[col].max()
    product_pdf[col] = (product_pdf[col] - cmin) / (cmax - cmin + 1e-8)

for col in ["LAT", "LON"]:
    cmin, cmax = store_pdf[col].min(), store_pdf[col].max()
    store_pdf[col] = (store_pdf[col] - cmin) / (cmax - cmin + 1e-8)

# Build node feature tensors (exclude ID column)
product_x = torch.tensor(
    product_pdf.drop(columns=["PRODUCT_ID"]).values, dtype=torch.float32
)
store_x = torch.tensor(
    store_pdf.drop(columns=["STORE_ID"]).values, dtype=torch.float32
)

print(f"Product node features: {product_x.shape}")  # [200, 6]
print(f"Store node features:   {store_x.shape}")    # [50, 5]

In [ ]:
# Build edge index tensors (remap IDs to 0-based indices)
# Product IDs are 1-200 -> index 0-199
# Store IDs are 1-50 -> index 0-49

# Product-Product edges
pp_edge_index = torch.tensor(
    np.array([
        pp_edges_pdf["SRC_PRODUCT_ID"].values - 1,
        pp_edges_pdf["DST_PRODUCT_ID"].values - 1,
    ]),
    dtype=torch.long,
)
pp_edge_weight = torch.tensor(pp_edges_pdf["WEIGHT"].values, dtype=torch.float32)

# Store-Store edges
ss_edge_index = torch.tensor(
    np.array([
        ss_edges_pdf["SRC_STORE_ID"].values - 1,
        ss_edges_pdf["DST_STORE_ID"].values - 1,
    ]),
    dtype=torch.long,
)
ss_edge_weight = torch.tensor(ss_edges_pdf["WEIGHT"].values, dtype=torch.float32)

# Product-Store edges (bipartite)
ps_edge_index = torch.tensor(
    np.array([
        ps_edges_pdf["PRODUCT_ID"].values - 1,
        ps_edges_pdf["STORE_ID"].values - 1,
    ]),
    dtype=torch.long,
)
ps_edge_weight = torch.tensor(ps_edges_pdf["WEIGHT"].values, dtype=torch.float32)

# Reverse Product-Store edges (store -> product)
sp_edge_index = torch.stack([ps_edge_index[1], ps_edge_index[0]])
sp_edge_weight = ps_edge_weight.clone()

# Store everything in a plain dict (no PyG dependency)
graph_data = {
    "product_x": product_x,
    "store_x": store_x,
    "pp_edge_index": pp_edge_index,
    "pp_edge_weight": pp_edge_weight,
    "ss_edge_index": ss_edge_index,
    "ss_edge_weight": ss_edge_weight,
    "ps_edge_index": ps_edge_index,
    "ps_edge_weight": ps_edge_weight,
    "sp_edge_index": sp_edge_index,
    "sp_edge_weight": sp_edge_weight,
}

n_products = product_x.shape[0]
n_stores = store_x.shape[0]
total_edges = pp_edge_index.shape[1] + ss_edge_index.shape[1] + ps_edge_index.shape[1] + sp_edge_index.shape[1]
print(f"Nodes: {n_products} products + {n_stores} stores = {n_products + n_stores}")
print(f"Edges: {pp_edge_index.shape[1]} p-p + {ss_edge_index.shape[1]} s-s + {ps_edge_index.shape[1]} p->s + {sp_edge_index.shape[1]} s->p = {total_edges}")

In [ ]:
# Serialize graph data for use inside distributed training function
# The graph is small (~250 nodes, ~10K edges) so serialization is efficient
graph_bytes = pickle.dumps(graph_data)

# Save to /tmp so the training function can load it
with open("/tmp/graph_data.pkl", "wb") as f:
    f.write(graph_bytes)

print(f"Graph serialized: {len(graph_bytes) / 1024:.1f} KB")

## 5. GNN Model Definition

A heterogeneous Graph Neural Network for dynamic pricing prediction (pure PyTorch, no PyG):
- **2 SAGE-style conv layers** using `scatter_add` message passing across 4 edge types
- **Product and store embeddings** are concatenated for each product-store pair
- **MLP head** maps concatenated embeddings to a price prediction
- **Loss:** MSE on selling price regression

In [ ]:
class SAGEConvLayer(nn.Module):
    """SAGE-style convolution using scatter_add (pure PyTorch).

    For each edge (src -> dst), we transform the source node features,
    aggregate them at the destination via scatter_add, then combine
    with the destination's own features.
    """

    def __init__(self, in_channels_src: int, in_channels_dst: int, out_channels: int):
        super().__init__()
        self.lin_src = nn.Linear(in_channels_src, out_channels)
        self.lin_dst = nn.Linear(in_channels_dst, out_channels)

    def forward(
        self,
        x_src: torch.Tensor,
        x_dst: torch.Tensor,
        edge_index: torch.Tensor,
        edge_weight: torch.Tensor,
        num_dst_nodes: int,
    ) -> torch.Tensor:
        """Forward pass.

        Args:
            x_src: Source node features [N_src, in_src]
            x_dst: Destination node features [N_dst, in_dst]
            edge_index: [2, E] with row 0 = src indices, row 1 = dst indices
            edge_weight: [E] scalar weights per edge
            num_dst_nodes: number of destination nodes (for scatter dim)

        Returns:
            Updated destination node features [N_dst, out]
        """
        src_idx = edge_index[0]  # [E]
        dst_idx = edge_index[1]  # [E]

        # Transform source features and weight by edge
        msg = self.lin_src(x_src[src_idx])  # [E, out]
        msg = msg * edge_weight.unsqueeze(-1)  # [E, out]

        # Aggregate at destinations via scatter_add
        agg = torch.zeros(num_dst_nodes, msg.size(-1), device=msg.device)
        agg.scatter_add_(0, dst_idx.unsqueeze(-1).expand_as(msg), msg)

        # Combine with destination self-features
        out = agg + self.lin_dst(x_dst)
        return out


class DynamicPricingGNN(nn.Module):
    """Heterogeneous GNN for dynamic pricing (pure PyTorch, no PyG).

    Message passing over 4 edge types:
      - product <-> product (substitutes, complements, same category)
      - store <-> store (geographic proximity)
      - product -> store, store -> product (product-store affinity)

    Each layer applies SAGEConvLayer per edge type and sums the
    contributions at each destination node type.
    """

    def __init__(
        self,
        product_in_channels: int,
        store_in_channels: int,
        hidden_channels: int = 64,
        num_layers: int = 2,
    ):
        super().__init__()
        self.num_layers = num_layers
        self.hidden = hidden_channels

        # Input projections
        self.product_lin = nn.Linear(product_in_channels, hidden_channels)
        self.store_lin = nn.Linear(store_in_channels, hidden_channels)

        # Per-layer conv modules for each edge type
        # pp = product->product, ss = store->store
        # ps = product->store,   sp = store->product
        self.pp_convs = nn.ModuleList()
        self.ss_convs = nn.ModuleList()
        self.ps_convs = nn.ModuleList()
        self.sp_convs = nn.ModuleList()
        for _ in range(num_layers):
            self.pp_convs.append(SAGEConvLayer(hidden_channels, hidden_channels, hidden_channels))
            self.ss_convs.append(SAGEConvLayer(hidden_channels, hidden_channels, hidden_channels))
            self.ps_convs.append(SAGEConvLayer(hidden_channels, hidden_channels, hidden_channels))
            self.sp_convs.append(SAGEConvLayer(hidden_channels, hidden_channels, hidden_channels))

        # MLP head: concat product + store embeddings -> price
        self.mlp = nn.Sequential(
            nn.Linear(hidden_channels * 2, hidden_channels),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_channels, hidden_channels // 2),
            nn.ReLU(),
            nn.Linear(hidden_channels // 2, 1),
        )

    def forward(
        self,
        graph: dict,
        product_ids: torch.Tensor,
        store_ids: torch.Tensor,
    ) -> torch.Tensor:
        """Forward pass.

        Args:
            graph: dict with keys product_x, store_x, pp_edge_index,
                   pp_edge_weight, ss_*, ps_*, sp_*
            product_ids: 0-based product indices for this batch [B]
            store_ids: 0-based store indices for this batch [B]

        Returns:
            Price predictions [B, 1]
        """
        product_h = torch.relu(self.product_lin(graph["product_x"]))
        store_h = torch.relu(self.store_lin(graph["store_x"]))
        n_products = product_h.size(0)
        n_stores = store_h.size(0)

        for i in range(self.num_layers):
            # Compute messages from each edge type
            pp_out = self.pp_convs[i](product_h, product_h, graph["pp_edge_index"], graph["pp_edge_weight"], n_products)
            ss_out = self.ss_convs[i](store_h, store_h, graph["ss_edge_index"], graph["ss_edge_weight"], n_stores)
            # product -> store: src=product, dst=store
            ps_out = self.ps_convs[i](product_h, store_h, graph["ps_edge_index"], graph["ps_edge_weight"], n_stores)
            # store -> product: src=store, dst=product
            sp_out = self.sp_convs[i](store_h, product_h, graph["sp_edge_index"], graph["sp_edge_weight"], n_products)

            # Sum contributions per node type and apply activation
            product_h = torch.relu(pp_out + sp_out)
            store_h = torch.relu(ss_out + ps_out)

        # Look up embeddings for the batch's product-store pairs
        product_emb = product_h[product_ids]  # [B, hidden]
        store_emb = store_h[store_ids]        # [B, hidden]

        # Concatenate and predict price
        combined = torch.cat([product_emb, store_emb], dim=-1)  # [B, 2*hidden]
        return self.mlp(combined)


# Quick shape test (CPU)
_model = DynamicPricingGNN(
    product_in_channels=product_x.shape[1],
    store_in_channels=store_x.shape[1],
    hidden_channels=64,
)
_pids = torch.tensor([0, 1, 2])
_sids = torch.tensor([0, 1, 2])
_out = _model(graph_data, _pids, _sids)
print(f"Model output shape: {_out.shape}")  # [3, 1]
print(f"Model parameters: {sum(p.numel() for p in _model.parameters()):,}")
del _model, _out

## 6. Distributed Training with PyTorchDistributor

Training follows the Snowflake distributed pattern:
1. `ShardedDataConnector` distributes tabular training rows across GPU workers
2. Each worker reconstructs the full graph (small, ~250 nodes) on its device
3. GNN forward pass uses the full graph; loss is computed on the worker's data shard
4. `DistributedDataParallel` synchronizes gradients across workers

In [ ]:
# Training hyperparameters
NUM_EPOCHS = 10
BATCH_SIZE = 256
LEARNING_RATE = 0.001
HIDDEN_CHANNELS = 64
NUM_GNN_LAYERS = 2
PRODUCT_IN_CHANNELS = product_x.shape[1]  # 6
STORE_IN_CHANNELS = store_x.shape[1]      # 5

print(f"Epochs: {NUM_EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Hidden channels: {HIDDEN_CHANNELS}")
print(f"GNN layers: {NUM_GNN_LAYERS}")
print(f"Product features: {PRODUCT_IN_CHANNELS}")
print(f"Store features: {STORE_IN_CHANNELS}")

In [ ]:
def train_func():
    """Distributed training function executed on each GPU worker."""
    import os
    import time
    import pickle
    import torch
    import torch.nn as nn
    import torch.distributed as dist
    from torch.nn.parallel import DistributedDataParallel as DDP
    from torch.utils.data import DataLoader
    from snowflake.ml.modeling.distributors.pytorch import get_context

    # --- Context & distributed setup ---
    context = get_context()
    rank = context.get_rank()
    world_size = context.get_world_size()
    model_dir = context.get_model_dir()

    dist.init_process_group("nccl", rank=rank, world_size=world_size)
    torch.manual_seed(42)
    device = torch.device(f"cuda:{rank}" if torch.cuda.is_available() else "cpu")

    # --- Reconstruct graph on this worker ---
    with open("/tmp/graph_data.pkl", "rb") as f:
        gd = pickle.load(f)

    # Move graph tensors to device
    graph_on_device = {k: v.to(device) for k, v in gd.items()}

    product_in_ch = gd["product_x"].shape[1]
    store_in_ch = gd["store_x"].shape[1]

    # --- Define model classes (must be defined inside train_func for DDP) ---
    class _SAGEConvLayer(nn.Module):
        def __init__(self, in_src, in_dst, out_ch):
            super().__init__()
            self.lin_src = nn.Linear(in_src, out_ch)
            self.lin_dst = nn.Linear(in_dst, out_ch)

        def forward(self, x_src, x_dst, edge_index, edge_weight, num_dst):
            src_idx, dst_idx = edge_index[0], edge_index[1]
            msg = self.lin_src(x_src[src_idx]) * edge_weight.unsqueeze(-1)
            agg = torch.zeros(num_dst, msg.size(-1), device=msg.device)
            agg.scatter_add_(0, dst_idx.unsqueeze(-1).expand_as(msg), msg)
            return agg + self.lin_dst(x_dst)

    class _DynamicPricingGNN(nn.Module):
        def __init__(self, product_in, store_in, hidden=64, n_layers=2):
            super().__init__()
            self.num_layers = n_layers
            self.product_lin = nn.Linear(product_in, hidden)
            self.store_lin = nn.Linear(store_in, hidden)
            self.pp_convs = nn.ModuleList()
            self.ss_convs = nn.ModuleList()
            self.ps_convs = nn.ModuleList()
            self.sp_convs = nn.ModuleList()
            for _ in range(n_layers):
                self.pp_convs.append(_SAGEConvLayer(hidden, hidden, hidden))
                self.ss_convs.append(_SAGEConvLayer(hidden, hidden, hidden))
                self.ps_convs.append(_SAGEConvLayer(hidden, hidden, hidden))
                self.sp_convs.append(_SAGEConvLayer(hidden, hidden, hidden))
            self.mlp = nn.Sequential(
                nn.Linear(hidden * 2, hidden),
                nn.ReLU(),
                nn.Dropout(0.2),
                nn.Linear(hidden, hidden // 2),
                nn.ReLU(),
                nn.Linear(hidden // 2, 1),
            )

        def forward(self, graph, product_ids, store_ids):
            product_h = torch.relu(self.product_lin(graph["product_x"]))
            store_h = torch.relu(self.store_lin(graph["store_x"]))
            n_p, n_s = product_h.size(0), store_h.size(0)
            for i in range(self.num_layers):
                pp_out = self.pp_convs[i](product_h, product_h, graph["pp_edge_index"], graph["pp_edge_weight"], n_p)
                ss_out = self.ss_convs[i](store_h, store_h, graph["ss_edge_index"], graph["ss_edge_weight"], n_s)
                ps_out = self.ps_convs[i](product_h, store_h, graph["ps_edge_index"], graph["ps_edge_weight"], n_s)
                sp_out = self.sp_convs[i](store_h, product_h, graph["sp_edge_index"], graph["sp_edge_weight"], n_p)
                product_h = torch.relu(pp_out + sp_out)
                store_h = torch.relu(ss_out + ps_out)
            product_emb = product_h[product_ids]
            store_emb = store_h[store_ids]
            return self.mlp(torch.cat([product_emb, store_emb], dim=-1))

    model = _DynamicPricingGNN(
        product_in=product_in_ch,
        store_in=store_in_ch,
        hidden=HIDDEN_CHANNELS,
        n_layers=NUM_GNN_LAYERS,
    ).to(device)
    ddp_model = DDP(model, device_ids=[rank])

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(ddp_model.parameters(), lr=LEARNING_RATE)

    # --- Load sharded tabular data ---
    dataset_map = context.get_dataset_map()
    train_pipe = dataset_map["train"].get_shard().to_torch_datapipe(
        batch_size=BATCH_SIZE, shuffle=True
    )
    val_pipe = dataset_map["val"].get_shard().to_torch_datapipe(
        batch_size=BATCH_SIZE, shuffle=False
    )
    train_loader = DataLoader(train_pipe, batch_size=None)
    val_loader = DataLoader(val_pipe, batch_size=None)

    # --- Training loop ---
    for epoch in range(NUM_EPOCHS):
        ddp_model.train()
        start_time = time.time()
        running_loss = 0.0
        n_batches = 0

        for batch_idx, batch_data in enumerate(train_loader):
            product_ids = (batch_data["PRODUCT_ID"].squeeze().long() - 1).to(device)
            store_ids = (batch_data["STORE_ID"].squeeze().long() - 1).to(device)
            target = batch_data["TARGET_PRICE"].squeeze().float().to(device)

            optimizer.zero_grad()
            pred = ddp_model(graph_on_device, product_ids, store_ids).squeeze()
            loss = criterion(pred, target)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            n_batches += 1

            if (batch_idx + 1) % 50 == 0:
                print(
                    f"[Rank {rank}] Epoch {epoch+1}/{NUM_EPOCHS}, "
                    f"Batch {batch_idx+1}: Loss={running_loss / n_batches:.4f}"
                )

        avg_train_loss = torch.tensor(running_loss / max(n_batches, 1), device=device)
        dist.all_reduce(avg_train_loss)
        avg_train_loss = avg_train_loss.item() / world_size

        # --- Validation ---
        ddp_model.eval()
        val_loss = 0.0
        val_batches = 0
        with torch.no_grad():
            for val_batch in val_loader:
                p_ids = (val_batch["PRODUCT_ID"].squeeze().long() - 1).to(device)
                s_ids = (val_batch["STORE_ID"].squeeze().long() - 1).to(device)
                target_val = val_batch["TARGET_PRICE"].squeeze().float().to(device)
                pred_val = ddp_model(graph_on_device, p_ids, s_ids).squeeze()
                val_loss += criterion(pred_val, target_val).item()
                val_batches += 1

        avg_val_loss = torch.tensor(val_loss / max(val_batches, 1), device=device)
        dist.all_reduce(avg_val_loss)
        avg_val_loss = avg_val_loss.item() / world_size

        dist.barrier()
        elapsed = time.time() - start_time

        if rank == 0:
            print(
                f"Epoch {epoch+1}/{NUM_EPOCHS} | "
                f"Train Loss: {avg_train_loss:.4f} | "
                f"Val Loss: {avg_val_loss:.4f} | "
                f"Time: {elapsed:.1f}s"
            )

    # --- Save model ---
    if rank == 0:
        save_path = os.path.join(model_dir, "dynamic_pricing_gnn.pth")
        torch.save(model.state_dict(), save_path)
        print(f"Model saved to {save_path}")

    dist.destroy_process_group()

In [ ]:
# Set up distributed trainer
pytorch_trainer = PyTorchDistributor(
    train_func=train_func,
    scaling_config=PyTorchScalingConfig(
        num_nodes=1,
        num_workers_per_node=1,
        resource_requirements_per_worker=WorkerResourceConfig(
            num_cpus=0, num_gpus=1
        ),
    ),
)

# Create sharded data connectors from training tables
train_data = session.table(f"{DB}.{DATA_SCHEMA}.TRAIN_DATA")
val_data = session.table(f"{DB}.{DATA_SCHEMA}.VAL_DATA")

data_train = ShardedDataConnector.from_dataframe(train_data)
data_val = ShardedDataConnector.from_dataframe(val_data)

print(f"Training rows: {train_data.count()}")
print(f"Validation rows: {val_data.count()}")
print("Launching distributed training...")

out = pytorch_trainer.run(
    dataset_map=dict(
        train=data_train,
        val=data_val,
    )
)
print("Training complete.")